In [0]:
# Definir pastas do projetos em variaveis para facilitar
bronze_path   = '/Volumes/bikestore/resource/bronze/'
silver_path   = '/Volumes/bikestore/resource/silver/'
gold_path     = '/Volumes/bikestore/resource/gold/'
resource_path = '/Workspace/Users/wesllan2000@yahoo.com.br/project_databricks_bikestore/resource/origem'


In [0]:
display(dbutils.fs.ls(resource_path))# map variavel

In [0]:
from pyspark.sql.functions import col, input_file_name, current_timestamp

df_brands=spark.read.csv(f'{resource_path}/brands.csv',
                         header=True,
                         inferSchema=True,
                         sep=','
                                              
                        ).withColumn("_input_path", col("_metadata.file_path"))\
                        .withColumn("_ingestion_date", current_timestamp())

In [0]:
from pyspark.sql.functions import col, current_timestamp

df_brands = (
    spark.read
    .format("csv")
    .options(header=True, inferSchema=True, sep=",")
    .load(f"{resource_path}/brands.csv")
    .select(
        "*",
        current_timestamp().alias("_ingestion_date"),
        col("_metadata.file_path").alias("_input_path"),
        
    )
)

In [0]:
display(df_brands)

In [0]:
#salvar em parquet como delta 
df_brands.write\
    .mode('overwrite')\
    .format('delta')\
    .option("mergeSchema", "true") \
    .save(f"{bronze_path}/brands")


In [0]:
%skip
--criar uma tabela com o external location 'nao iremos usar na bronze' pois sao os dados brutos e prescisam de tratamento (geralmente nao sao criadas tabelas pra esta parte , mas nao é uma regra)

CREATE TABLE IF NOT EXISTS bikestore.logistics.bronze_brands
LOCATION 'abfss://uc-ext-azure@externalazure.dfs.core.windows.net/bikestore/bronze/brands/'
